# 015 Torch training -- the loop, stripped bare

Conceptual lean version of `070_torch_training.py` (no animation).

One parameter, no bias:

$$
\hat y = w\, x
$$

Train `w` by full-batch gradient descent on MSE. Data sit near the line `y = 2x`.

| Piece | Role |
|-------|------|
| `model` | maps `x -> w*x` |
| `loss` | how wrong the prediction is |
| `backward` | fills `w.grad` |
| `step` | `w <- w - lr * w.grad` |

Synced with `0270_torch_training.py` (same code cells).


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(523)

N = 30
x = torch.linspace(1.0, 5.0, N).unsqueeze(1)
y = 2.0 * x + 0.15 * torch.randn(N, 1)  # mild noise; truth y=2x

model = nn.Linear(1, 1, bias=False)     # one weight w; y_hat = w * x
with torch.no_grad():
    model.weight.fill_(-0.5)            # start far from 2
loss_fn = nn.MSELoss()
# full-batch GD on all N rows (PyTorch API class name below)
opt = torch.optim.SGD(model.parameters(), lr=0.005)


## One training step, repeated

```text
predict  ->  loss  ->  zero_grad  ->  backward  ->  step
```

Here every training step uses **all** `N = 30` points (full-batch GD).

Update rule:

$$
w \leftarrow w - \mathrm{lr}\cdot g,\qquad g=\partial\ell/\partial w
$$


In [ ]:
STEPS = 150

for step in range(STEPS):
    y_hat = model(x)              # forward on ALL N points
    loss = loss_fn(y_hat, y)      # scalar MSE
    opt.zero_grad()
    loss.backward()               # compute d(loss)/dw
    opt.step()                    # w <- w - lr * grad

w = model.weight.item()
print(f"learned w = {w:.4f}   (true slope ~ 2)")
print(f"final MSE = {loss_fn(model(x), y).item():.4f}")
